# Pipeline **ELT** com **dbt** — Acidentes de Trânsito do Recife (2014–2016)

| Etapa | O que acontece | Ferramenta |
|------|----------------|-----------|
| **E**xtract | Leitura das 8 fontes (CSV + GeoJSON) | Colab |
| **L**oad | Dados **brutos**, sem transformação, em tabelas `raw` | DuckDB |
| **T**ransform | **Todo** o tratamento e a modelagem estrela são **modelos dbt** (SQL) executados dentro do warehouse | **dbt** + DuckDB |

> No ELT a transformação roda **depois** da carga, **dentro** do warehouse. O **dbt é o "T"**: cada tabela do modelo estrela é um *model* dbt, com testes de qualidade (`unique`, `not_null`, `relationships`) e DAG de dependências gerado pelos `ref()`.

**Projeto dbt gerado pelo notebook:**
```
dbt_acidentes/
├── dbt_project.yml
├── profiles.yml
├── macros/generate_schema_name.sql
└── models/
    ├── staging/  _sources.yml · stg_acidentes.sql
    └── marts/    dim_tempo · dim_local · dim_veiculo · dim_ocorrencia
                  dim_tipo_acidente · fato_acidente · _marts.yml (testes)
```

## Setup: instalar dbt + adapter DuckDB

In [1]:
!pip install -q dbt-duckdb
import duckdb, pandas as pd, os, glob, textwrap
!dbt --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.2/118.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.7/442.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.7/238.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## Upload dos arquivos de origem (8 arquivos)
`acidentes-2014.csv`, `acidentes-com-vitimas-ocorridos-no-ano-de-2015.csv`, `acidentes-abril-2015.csv`,
`acidentes-maio-2015.csv`, `acidentes-de-transito-com-vitimas-2016.csv`,
`acidentes-janeiro-2015.geojson`, `acidentes-fevereiro-2015.geojson`, `acidentes-marco-2015.geojson`.

In [2]:
# Opção A: upload manual (descomente)
# from google.colab import files; files.upload()

# Opção B: arquivos já em /content
BASE = '/content'                 # ajuste se necessário
DW_PATH = '/content/acidentes_dw.duckdb'
os.environ['DW_PATH'] = DW_PATH   # usado pelo profiles.yml do dbt
encontrados = sorted(glob.glob(f'{BASE}/acidentes-*'))
print(f'{len(encontrados)} arquivos encontrados:'); [print('  -', os.path.basename(f)) for f in encontrados]
assert len(encontrados) >= 8, 'Faltam arquivos! Suba os 8 arquivos de origem.'

8 arquivos encontrados:
  - acidentes-2014.csv
  - acidentes-abril-2015.csv
  - acidentes-com-vitimas-ocorridos-no-ano-de-2015.csv
  - acidentes-de-transito-com-vitimas-2016.csv
  - acidentes-fevereiro-2015.geojson
  - acidentes-janeiro-2015.geojson
  - acidentes-maio-2015.csv
  - acidentes-marco-2015.geojson


## Passo **L** (LOAD): dados brutos no schema `raw`
Carga dos dados

In [5]:
LOAD_SQL = r'''
/* =====================================================================
   ELT — PASSO L (LOAD): pousa os dados BRUTOS, exatamente como vêm da
   origem, em tabelas de staging (schema raw).
   Todas as colunas entram como texto; datas, números e regras de negócio
   só serão tratados depois, em SQL, no passo T.
   ===================================================================== */
CREATE SCHEMA IF NOT EXISTS raw;

CREATE OR REPLACE TABLE raw.acid_2014 AS
SELECT * FROM read_csv('{BASE}/acidentes-2014.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_vitimas AS
SELECT * FROM read_csv('{BASE}/acidentes-com-vitimas-ocorridos-no-ano-de-2015.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_abril AS
SELECT * FROM read_csv('{BASE}/acidentes-abril-2015.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2015_maio AS
SELECT * FROM read_csv('{BASE}/acidentes-maio-2015.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

CREATE OR REPLACE TABLE raw.acid_2016 AS
SELECT * FROM read_csv('{BASE}/acidentes-de-transito-com-vitimas-2016.csv',
       delim=';', header=true, all_varchar=true, quote='"', ignore_errors=true);

/* GeoJSON: desaninha o array features e pousa propriedades + coordenadas */
CREATE OR REPLACE TABLE raw.acid_2015_jan AS
SELECT feat.properties.tipo::VARCHAR AS tipo, feat.properties.data::VARCHAR AS data,
       feat.properties.detalhes::VARCHAR AS detalhes,
       feat.properties.latitude::VARCHAR AS latitude, feat.properties.longitude::VARCHAR AS longitude
FROM (SELECT unnest(features) AS feat FROM read_json('{BASE}/acidentes-janeiro-2015.geojson'));

CREATE OR REPLACE TABLE raw.acid_2015_fev AS
SELECT feat.properties.tipo::VARCHAR AS tipo, feat.properties.data::VARCHAR AS data,
       feat.properties.detalhes::VARCHAR AS detalhes,
       feat.properties.latitude::VARCHAR AS latitude, feat.properties.longitude::VARCHAR AS longitude
FROM (SELECT unnest(features) AS feat FROM read_json('{BASE}/acidentes-fevereiro-2015.geojson'));

CREATE OR REPLACE TABLE raw.acid_2015_mar AS
SELECT feat.properties.tipo::VARCHAR AS tipo, feat.properties.data::VARCHAR AS data,
       feat.properties.detalhes::VARCHAR AS detalhes,
       feat.properties.latitude::VARCHAR AS latitude, feat.properties.longitude::VARCHAR AS longitude
FROM (SELECT unnest(features) AS feat FROM read_json('{BASE}/acidentes-marco-2015.geojson'));

'''
con = duckdb.connect(DW_PATH)
con.execute(LOAD_SQL.replace('{BASE}', BASE))
print('Tabelas brutas (schema raw):')
print(con.execute("SELECT table_name, estimated_size FROM duckdb_tables() WHERE schema_name='raw' ORDER BY 1").df())
con.close()   # libera o arquivo para o dbt
print('Pronto para o dbt.')

Tabelas brutas (schema raw):
          table_name  estimated_size
0          acid_2014            1734
1    acid_2015_abril             184
2      acid_2015_fev             162
3      acid_2015_jan             188
4     acid_2015_maio             146
5      acid_2015_mar             189
6  acid_2015_vitimas            1369
7          acid_2016            1231
Pronto para o dbt.


## Materializar o projeto **dbt** em disco
A célula abaixo escreve todos os arquivos do projeto dbt (configuração, profile, macro, sources, models e testes).

In [6]:
PROJECT_FILES = {
 "profiles.yml": "acidentes:\n  target: dev\n  outputs:\n    dev:\n      type: duckdb\n      path: \"{{ env_var('DW_PATH', 'acidentes_dw.duckdb') }}\"\n      threads: 4\n",
 ".user.yml": "id: 4f64c6ed-8d8a-49b8-8a16-33cd011ccdc9\n",
 "dbt_project.yml": "name: 'acidentes_recife'\nversion: '1.0.0'\nprofile: 'acidentes'\nconfig-version: 2\n\nmodel-paths: [\"models\"]\nmacro-paths: [\"macros\"]\ntarget-path: \"target\"\nclean-targets: [\"target\", \"dbt_packages\"]\n\nmodels:\n  acidentes_recife:\n    staging:\n      +schema: stg          # camada de staging -> schema stg\n      +materialized: table\n    marts:\n      +schema: dw           # esquema estrela -> schema dw\n      +materialized: table\n",
 "macros/generate_schema_name.sql": "{% macro generate_schema_name(custom_schema_name, node) -%}\n    {%- if custom_schema_name is none -%}\n        {{ target.schema }}\n    {%- else -%}\n        {{ custom_schema_name | trim }}\n    {%- endif -%}\n{%- endmacro %}\n",
 "models/marts/dim_veiculo.sql": "WITH classif AS (\n  SELECT DISTINCT\n    CASE\n      WHEN tipo_raw LIKE 'MOTO%' THEN 'Motocicleta'\n      WHEN tipo_raw LIKE 'CICLOMOT%' THEN 'Ciclomotor'\n      WHEN tipo_raw LIKE 'AUTOM%' OR tipo_raw LIKE 'COLIS%' THEN 'Automóvel'\n      WHEN tipo_raw LIKE 'CICLISTA%' OR tipo_raw LIKE 'PEDESTRES E CICLISTA%' THEN 'Ciclista'\n      WHEN tipo_raw LIKE 'PEDESTRE%' OR tipo_raw LIKE 'ATROPELAMENTO%' THEN 'Pedestre'\n      ELSE 'Outros'\n    END AS tipo_veiculo\n  FROM {{ ref('stg_acidentes') }}\n)\nSELECT ROW_NUMBER() OVER (ORDER BY tipo_veiculo) AS sk_veiculo,\n       tipo_veiculo,\n       CASE tipo_veiculo\n         WHEN 'Motocicleta' THEN 'Motorizado de duas rodas'\n         WHEN 'Ciclomotor'  THEN 'Motorizado de duas rodas'\n         WHEN 'Automóvel'   THEN 'Motorizado de quatro ou mais rodas'\n         WHEN 'Ciclista'    THEN 'Não motorizado'\n         WHEN 'Pedestre'    THEN 'Pedestre'\n         ELSE 'Outros'\n       END AS categoria_veiculo\nFROM classif\n",
 "models/marts/dim_ocorrencia.sql": "WITH base AS (\n  SELECT DISTINCT COALESCE(NULLIF(ocorrencia_raw,''),'NÃO INFORMADO') AS tipo_ocorrencia\n  FROM {{ ref('stg_acidentes') }}\n)\nSELECT ROW_NUMBER() OVER (ORDER BY tipo_ocorrencia) AS sk_ocorrencia,\n       tipo_ocorrencia,\n       CASE\n         WHEN tipo_ocorrencia LIKE 'COLIS%' OR tipo_ocorrencia LIKE 'CHOQUE%'\n              OR tipo_ocorrencia LIKE 'ENGAVET%' THEN 'Colisão'\n         WHEN tipo_ocorrencia LIKE 'ATROPELAMENTO ANIMAL%' THEN 'Atropelamento de animal'\n         WHEN tipo_ocorrencia LIKE 'ATROPELAMENTO%' THEN 'Atropelamento'\n         WHEN tipo_ocorrencia LIKE 'CAPOTAMENTO%' OR tipo_ocorrencia LIKE 'TOMBAMENTO%'\n              THEN 'Perda de controle'\n         ELSE 'Outros'\n       END AS classificacao_ocorrencia\nFROM base\n",
 "models/marts/dim_tempo.sql": "SELECT ROW_NUMBER() OVER (ORDER BY data_acidente, hora_acidente) AS sk_tempo,\n       data_acidente,\n       EXTRACT(year  FROM data_acidente) AS ano,\n       EXTRACT(month FROM data_acidente) AS mes,\n       EXTRACT(day   FROM data_acidente) AS dia,\n       hora_acidente,\n       CASE EXTRACT(dow FROM data_acidente)\n            WHEN 0 THEN 'Domingo' WHEN 1 THEN 'Segunda-feira' WHEN 2 THEN 'Terça-feira'\n            WHEN 3 THEN 'Quarta-feira' WHEN 4 THEN 'Quinta-feira' WHEN 5 THEN 'Sexta-feira'\n            WHEN 6 THEN 'Sábado' END AS dia_semana,\n       CASE WHEN EXTRACT(dow FROM data_acidente) IN (0,6) THEN 'Sim' ELSE 'Não' END AS fim_de_semana\nFROM (SELECT DISTINCT data_acidente, hora_acidente FROM {{ ref('stg_acidentes') }})\n",
 "models/marts/_marts.yml": "version: 2\n\nmodels:\n  - name: dim_tempo\n    columns:\n      - name: sk_tempo\n        tests: [unique, not_null]\n      - name: data_acidente\n        tests: [not_null]\n  - name: dim_local\n    columns:\n      - name: sk_local\n        tests: [unique, not_null]\n  - name: dim_veiculo\n    columns:\n      - name: sk_veiculo\n        tests: [unique, not_null]\n      - name: tipo_veiculo\n        tests: [not_null]\n  - name: dim_ocorrencia\n    columns:\n      - name: sk_ocorrencia\n        tests: [unique, not_null]\n  - name: dim_tipo_acidente\n    columns:\n      - name: sk_tipo_acidente\n        tests: [unique, not_null]\n  - name: fato_acidente\n    columns:\n      - name: sk_acidente\n        tests: [unique, not_null]\n      - name: sk_tempo\n        tests:\n          - not_null\n          - relationships: {to: ref('dim_tempo'), field: sk_tempo}\n      - name: sk_local\n        tests:\n          - not_null\n          - relationships: {to: ref('dim_local'), field: sk_local}\n      - name: sk_veiculo\n        tests:\n          - not_null\n          - relationships: {to: ref('dim_veiculo'), field: sk_veiculo}\n      - name: sk_ocorrencia\n        tests:\n          - not_null\n          - relationships: {to: ref('dim_ocorrencia'), field: sk_ocorrencia}\n      - name: sk_tipo_acidente\n        tests:\n          - not_null\n          - relationships: {to: ref('dim_tipo_acidente'), field: sk_tipo_acidente}\n      - name: quantidade_vitimas\n        tests: [not_null]\n",
 "models/marts/dim_tipo_acidente.sql": "WITH base AS (\n  SELECT DISTINCT\n    CASE\n      WHEN ocorrencia_raw LIKE 'ATROPELAMENTO ANIMAL%' OR descricao_detalhada LIKE '%ANIMAL%'\n           THEN 'Atropelamento de animal'\n      WHEN ocorrencia_raw LIKE 'ATROPELAMENTO%' OR tipo_raw LIKE 'PEDESTRE%'\n           OR tipo_raw LIKE 'ATROPELAMENTO%' THEN 'Atropelamento de pedestre'\n      WHEN ocorrencia_raw LIKE 'CAPOTAMENTO%' OR ocorrencia_raw LIKE 'TOMBAMENTO%'\n           THEN 'Capotamento/Tombamento'\n      WHEN ocorrencia_raw LIKE 'COLIS%' OR ocorrencia_raw LIKE 'CHOQUE%'\n           OR ocorrencia_raw LIKE 'ENGAVET%' OR tipo_raw LIKE 'COLIS%'\n           THEN 'Colisão entre veículos'\n      WHEN tipo_raw LIKE 'CICLISTA%' THEN 'Acidente com ciclista'\n      ELSE 'Não classificado'\n    END AS causa_acidente,\n    descricao_detalhada\n  FROM {{ ref('stg_acidentes') }}\n)\nSELECT ROW_NUMBER() OVER (ORDER BY causa_acidente, descricao_detalhada) AS sk_tipo_acidente,\n       causa_acidente, descricao_detalhada\nFROM base\n",
 "models/marts/fato_acidente.sql": "/* FATO: junta o staging às dimensões (via ref) pelas chaves naturais p/ obter as FKs.\n   O dbt monta o DAG automaticamente a partir dos ref(). */\nWITH enriquecido AS (\n  SELECT\n    s.*,\n    CASE\n      WHEN s.tipo_raw LIKE 'MOTO%' THEN 'Motocicleta'\n      WHEN s.tipo_raw LIKE 'CICLOMOT%' THEN 'Ciclomotor'\n      WHEN s.tipo_raw LIKE 'AUTOM%' OR s.tipo_raw LIKE 'COLIS%' THEN 'Automóvel'\n      WHEN s.tipo_raw LIKE 'CICLISTA%' OR s.tipo_raw LIKE 'PEDESTRES E CICLISTA%' THEN 'Ciclista'\n      WHEN s.tipo_raw LIKE 'PEDESTRE%' OR s.tipo_raw LIKE 'ATROPELAMENTO%' THEN 'Pedestre'\n      ELSE 'Outros'\n    END AS tipo_veiculo,\n    COALESCE(NULLIF(s.ocorrencia_raw,''),'NÃO INFORMADO') AS tipo_ocorrencia,\n    CASE\n      WHEN s.ocorrencia_raw LIKE 'ATROPELAMENTO ANIMAL%' OR s.descricao_detalhada LIKE '%ANIMAL%' THEN 'Atropelamento de animal'\n      WHEN s.ocorrencia_raw LIKE 'ATROPELAMENTO%' OR s.tipo_raw LIKE 'PEDESTRE%' OR s.tipo_raw LIKE 'ATROPELAMENTO%' THEN 'Atropelamento de pedestre'\n      WHEN s.ocorrencia_raw LIKE 'CAPOTAMENTO%' OR s.ocorrencia_raw LIKE 'TOMBAMENTO%' THEN 'Capotamento/Tombamento'\n      WHEN s.ocorrencia_raw LIKE 'COLIS%' OR s.ocorrencia_raw LIKE 'CHOQUE%' OR s.ocorrencia_raw LIKE 'ENGAVET%' OR s.tipo_raw LIKE 'COLIS%' THEN 'Colisão entre veículos'\n      WHEN s.tipo_raw LIKE 'CICLISTA%' THEN 'Acidente com ciclista'\n      ELSE 'Não classificado'\n    END AS causa_acidente\n  FROM {{ ref('stg_acidentes') }} s\n)\nSELECT\n   ROW_NUMBER() OVER (ORDER BY e.data_acidente, e.fonte) AS sk_acidente,\n   t.sk_tempo, l.sk_local, ta.sk_tipo_acidente, o.sk_ocorrencia, v.sk_veiculo,\n   e.quantidade_vitimas,\n   1 AS quantidade_veiculos\nFROM enriquecido e\nJOIN {{ ref('dim_tempo') }} t  ON t.data_acidente = e.data_acidente\n                              AND t.hora_acidente IS NOT DISTINCT FROM e.hora_acidente\nJOIN {{ ref('dim_local') }} l  ON l.bairro=e.bairro AND l.endereco=e.endereco\n                              AND l.latitude IS NOT DISTINCT FROM e.latitude\n                              AND l.longitude IS NOT DISTINCT FROM e.longitude\nJOIN {{ ref('dim_veiculo') }} v ON v.tipo_veiculo = e.tipo_veiculo\nJOIN {{ ref('dim_ocorrencia') }} o ON o.tipo_ocorrencia = e.tipo_ocorrencia\nJOIN {{ ref('dim_tipo_acidente') }} ta ON ta.causa_acidente = e.causa_acidente\n                                       AND ta.descricao_detalhada = e.descricao_detalhada\n",
 "models/marts/dim_local.sql": "SELECT ROW_NUMBER() OVER (ORDER BY bairro, endereco) AS sk_local,\n       bairro, endereco, latitude, longitude\nFROM (SELECT DISTINCT bairro, endereco, latitude, longitude FROM {{ ref('stg_acidentes') }})\n",
 "models/staging/_sources.yml": "version: 2\n\nsources:\n  - name: raw\n    description: \"Dados brutos das 8 fontes, pousados sem transformação (passo LOAD do ELT).\"\n    schema: raw\n    tables:\n      - name: acid_2014\n      - name: acid_2015_vitimas\n      - name: acid_2015_abril\n      - name: acid_2015_maio\n      - name: acid_2016\n      - name: acid_2015_jan\n      - name: acid_2015_fev\n      - name: acid_2015_mar\n",
 "models/staging/stg_acidentes.sql": "/* STAGING: integra as 8 fontes heterogêneas, converte datas (3 formatos),\n   trata lat/long, padroniza texto, limpa sujeira e filtra a janela 2014–2016.\n   É o passo T do ELT, orquestrado pelo dbt. */\nWITH unificado AS (\n    SELECT 'acidentes-2014' AS fonte,\n           try_strptime(data,'%-m/%-d/%Y') AS dt,\n           NULL::VARCHAR AS hora_raw,\n           NULL::VARCHAR AS bairro, NULL::VARCHAR AS endereco, NULL::VARCHAR AS complemento,\n           NULL::VARCHAR AS ocorrencia_raw, NULL::VARCHAR AS vitimas_raw,\n           detalhes AS descricao, tipo AS tipo_raw, detalhes AS detalhes_raw,\n           latitude AS lat_raw, longitude AS lon_raw\n    FROM {{ source('raw','acid_2014') }}\n  UNION ALL\n    SELECT 'acidentes-abril-2015', try_strptime(data,'%Y-%m-%d'), NULL,\n           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude\n    FROM {{ source('raw','acid_2015_abril') }}\n  UNION ALL\n    SELECT 'acidentes-maio-2015', try_strptime(data,'%Y-%m-%d'), NULL,\n           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude\n    FROM {{ source('raw','acid_2015_maio') }}\n  UNION ALL\n    SELECT 'acidentes-janeiro-2015', try_strptime(data,'%d/%m/%Y'), NULL,\n           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude\n    FROM {{ source('raw','acid_2015_jan') }}\n  UNION ALL\n    SELECT 'acidentes-fevereiro-2015', try_strptime(data,'%d/%m/%Y'), NULL,\n           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude\n    FROM {{ source('raw','acid_2015_fev') }}\n  UNION ALL\n    SELECT 'acidentes-marco-2015', try_strptime(data,'%d/%m/%Y'), NULL,\n           NULL,NULL,NULL, NULL, NULL, detalhes, tipo, detalhes, latitude, longitude\n    FROM {{ source('raw','acid_2015_mar') }}\n  UNION ALL\n    SELECT 'acidentes-com-vitimas-2015',\n           COALESCE(try_strptime(data_abertura,'%d/%m/%Y'), try_strptime(data_abertura,'%d/%m/%y')),\n           hora_abertura, bairro, endereco, complemento, tipo_ocorrencia, quantidade_vitimas,\n           descricao, tipo, descricao, latitude, longitude\n    FROM {{ source('raw','acid_2015_vitimas') }}\n  UNION ALL\n    SELECT 'acidentes-2016',\n           COALESCE(try_strptime(\"data de abertura\",'%d/%m/%Y'), try_strptime(\"data de abertura\",'%d/%m/%y')),\n           \"hora de abertura\", bairro, endereco, complemento, \"tipo de ocorrencia\", \"quantidade de vitimas\",\n           descricao, tipo, descricao, latitude, longitude\n    FROM {{ source('raw','acid_2016') }}\n)\nSELECT\n    fonte,\n    CAST(dt AS DATE) AS data_acidente,\n    CASE WHEN hora_raw ~ '^[0-2]?[0-9]:[0-5][0-9]'\n         THEN try_cast(strptime(trim(hora_raw),'%H:%M') AS TIME) END AS hora_acidente,\n    COALESCE(NULLIF(upper(trim(bairro)),''),'NÃO INFORMADO') AS bairro,\n    COALESCE(NULLIF(upper(trim(regexp_replace(endereco,'\\s+',' ','g'))),''),'NÃO INFORMADO') AS endereco,\n    CASE WHEN abs(try_cast(replace(lat_raw,',','.') AS DOUBLE))>20\n         THEN try_cast(replace(lon_raw,',','.') AS DOUBLE)\n         ELSE try_cast(replace(lat_raw,',','.') AS DOUBLE) END AS latitude,\n    CASE WHEN abs(try_cast(replace(lat_raw,',','.') AS DOUBLE))>20\n         THEN try_cast(replace(lat_raw,',','.') AS DOUBLE)\n         ELSE try_cast(replace(lon_raw,',','.') AS DOUBLE) END AS longitude,\n    CASE WHEN ocorrencia_raw IS NULL THEN NULL\n         ELSE regexp_replace(upper(trim(regexp_replace(ocorrencia_raw,'[\\t\"].*$',''))),'A+$','') END AS ocorrencia_raw,\n    GREATEST(COALESCE(try_cast(vitimas_raw AS INTEGER),1),1) AS quantidade_vitimas,\n    upper(trim(tipo_raw)) AS tipo_raw,\n    COALESCE(NULLIF(trim(descricao),''),'NÃO INFORMADO') AS descricao_detalhada\nFROM unificado\nWHERE dt IS NOT NULL\n  AND dt >= TIMESTAMP '2014-01-01'\n  AND dt <  TIMESTAMP '2017-01-01'\n"
}

import os
ROOT='dbt_acidentes'
for rel, content in PROJECT_FILES.items():
    dest=os.path.join(ROOT, rel); os.makedirs(os.path.dirname(dest), exist_ok=True)
    open(dest,'w',encoding='utf-8').write(content)
print('Projeto dbt criado:')
for rel in sorted(PROJECT_FILES): print('  ', os.path.join(ROOT, rel))

Projeto dbt criado:
   dbt_acidentes/.user.yml
   dbt_acidentes/dbt_project.yml
   dbt_acidentes/macros/generate_schema_name.sql
   dbt_acidentes/models/marts/_marts.yml
   dbt_acidentes/models/marts/dim_local.sql
   dbt_acidentes/models/marts/dim_ocorrencia.sql
   dbt_acidentes/models/marts/dim_tempo.sql
   dbt_acidentes/models/marts/dim_tipo_acidente.sql
   dbt_acidentes/models/marts/dim_veiculo.sql
   dbt_acidentes/models/marts/fato_acidente.sql
   dbt_acidentes/models/staging/_sources.yml
   dbt_acidentes/models/staging/stg_acidentes.sql
   dbt_acidentes/profiles.yml


## `dbt debug` (valida conexão e configuração)

In [7]:
!cd dbt_acidentes && dbt debug --profiles-dir .

19:25:40  Running with dbt=1.11.11
19:25:40  dbt version: 1.11.11
19:25:40  python version: 3.12.13
19:25:40  python path: /usr/bin/python3
19:25:40  os info: Linux-6.6.122+-x86_64-with-glibc2.35
19:25:40  Using profiles dir at .
19:25:40  Using profiles.yml file at ./profiles.yml
19:25:40  Using dbt_project.yml file at /content/dbt_acidentes/dbt_project.yml
19:25:40  adapter type: duckdb
19:25:40  adapter version: 1.10.1
19:25:40  Configuration:
19:25:40    profiles.yml file [OK found and valid]
19:25:40    dbt_project.yml file [OK found and valid]
19:25:40  Required dependencies:
19:25:40   - git [OK found]

19:25:40  Connection:
19:25:40    database: acidentes_dw
19:25:40    schema: main
19:25:40    path: /content/acidentes_dw.duckdb
19:25:40    config_options: None
19:25:40    extensions: None
19:25:40    settings: {}
19:25:40    external_root: .
19:25:40    use_credential_provider: None
19:25:40    attach: None
19:25:40    filesystems: None
19:25:40    remote: None
19:25:40    plu

## Passo **T** (TRANSFORM): `dbt run`
O dbt monta o DAG a partir dos `ref()`/`source()` e materializa, na ordem correta:
`stg_acidentes` → 5 dimensões → `fato_acidente`.

In [8]:
!cd dbt_acidentes && dbt run --profiles-dir .

19:26:02  Running with dbt=1.11.11
19:26:02  Registered adapter: duckdb=1.10.1
19:26:02  Unable to do partial parsing because saved manifest not found. Starting full parse.
19:26:05  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `relationships` defined on 'fato_acidente' in
package 'acidentes_recife' (models/marts/_marts.yml). Arguments to generic tests
should be nested under the `arguments` property.
19:26:05  Found 7 models, 25 data tests, 8 sources, 486 macros
19:26:05  
19:26:05  Concurrency: 4 threads (target='dev')
19:26:05  
19:26:05  1 of 7 START sql table model stg.stg_acidentes ................................. [RUN]
19:26:05  1 of 7 OK created sql table model stg.stg_acidentes ............................ [OK in 0.18s]
19:26:05  2 of 7 START sql table model dw.dim_local ...................................... [RUN]
19:26:05  3 of 7 START sql table model dw.dim_ocorrencia ................................

## Qualidade de dados: `dbt test`
Executa `unique`, `not_null` e `relationships` (integridade referencial das FKs).

In [9]:
!cd dbt_acidentes && dbt test --profiles-dir .

19:26:31  Running with dbt=1.11.11
19:26:31  Registered adapter: duckdb=1.10.1
19:26:32  Found 7 models, 25 data tests, 8 sources, 486 macros
19:26:32  
19:26:32  Concurrency: 4 threads (target='dev')
19:26:32  
19:26:33  1 of 25 START test not_null_dim_local_sk_local ................................. [RUN]
19:26:33  2 of 25 START test not_null_dim_ocorrencia_sk_ocorrencia ....................... [RUN]
19:26:33  3 of 25 START test not_null_dim_tempo_data_acidente ............................ [RUN]
19:26:33  4 of 25 START test not_null_dim_tempo_sk_tempo ................................. [RUN]
19:26:33  1 of 25 PASS not_null_dim_local_sk_local ....................................... [PASS in 0.32s]
19:26:33  2 of 25 PASS not_null_dim_ocorrencia_sk_ocorrencia ............................. [PASS in 0.32s]
19:26:33  3 of 25 PASS not_null_dim_tempo_data_acidente .................................. [PASS in 0.31s]
19:26:33  5 of 25 START test not_null_dim_tipo_acidente_sk_tipo_acidente ......

## Validação e amostra do esquema estrela

In [10]:
con = duckdb.connect(DW_PATH)
df = lambda q: con.execute(q).df()
for t in ['stg.stg_acidentes','dw.dim_tempo','dw.dim_local','dw.dim_veiculo',
          'dw.dim_ocorrencia','dw.dim_tipo_acidente','dw.fato_acidente']:
    print(f'  {t:<24}', con.execute(f'SELECT count(*) FROM {t}').fetchone()[0])

display(df('''
SELECT f.sk_acidente, t.data_acidente, t.dia_semana, t.fim_de_semana, l.bairro,
       v.tipo_veiculo, o.tipo_ocorrencia, ta.causa_acidente, f.quantidade_vitimas
FROM dw.fato_acidente f
JOIN dw.dim_tempo t USING(sk_tempo)
JOIN dw.dim_local l USING(sk_local)
JOIN dw.dim_veiculo v USING(sk_veiculo)
JOIN dw.dim_ocorrencia o USING(sk_ocorrencia)
JOIN dw.dim_tipo_acidente ta USING(sk_tipo_acidente)
LIMIT 10'''))

  stg.stg_acidentes        5191
  dw.dim_tempo             2995
  dw.dim_local             5189
  dw.dim_veiculo           6
  dw.dim_ocorrencia        14
  dw.dim_tipo_acidente     1195
  dw.fato_acidente         5191


,sk_acidente,data_acidente,dia_semana,fim_de_semana,bairro,tipo_veiculo,tipo_ocorrencia,causa_acidente,quantidade_vitimas
0,1,2014-02-24,Segunda-feira,Não,NÃO INFORMADO,Automóvel,NÃO INFORMADO,Colisão entre veículos,1
1,2,2014-02-25,Terça-feira,Não,NÃO INFORMADO,Motocicleta,NÃO INFORMADO,Não classificado,1
2,3,2014-02-28,Sexta-feira,Não,NÃO INFORMADO,Automóvel,NÃO INFORMADO,Colisão entre veículos,1
3,4,2014-03-01,Sábado,Sim,NÃO INFORMADO,Ciclista,NÃO INFORMADO,Acidente com ciclista,1
4,5,2014-03-01,Sábado,Sim,NÃO INFORMADO,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1
5,6,2014-03-02,Domingo,Sim,NÃO INFORMADO,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1
6,7,2014-03-03,Segunda-feira,Não,NÃO INFORMADO,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1
7,8,2014-03-04,Terça-feira,Não,NÃO INFORMADO,Ciclista,NÃO INFORMADO,Acidente com ciclista,1
8,9,2014-03-04,Terça-feira,Não,NÃO INFORMADO,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1
9,10,2014-03-04,Terça-feira,Não,NÃO INFORMADO,Pedestre,NÃO INFORMADO,Atropelamento de pedestre,1


## Exemplos de análise

In [11]:
display(df('''
SELECT t.ano, v.tipo_veiculo, count(*) AS qtd_acidentes, sum(f.quantidade_vitimas) AS total_vitimas
FROM dw.fato_acidente f
JOIN dw.dim_tempo t USING(sk_tempo)
JOIN dw.dim_veiculo v USING(sk_veiculo)
GROUP BY t.ano, v.tipo_veiculo ORDER BY t.ano, qtd_acidentes DESC'''))

,ano,tipo_veiculo,qtd_acidentes,total_vitimas
0,2014,Automóvel,754,754.0
1,2014,Motocicleta,629,629.0
2,2014,Pedestre,178,178.0
3,2014,Ciclista,150,150.0
4,2014,Ciclomotor,23,23.0
5,2014,Outros,3,3.0
6,2015,Motocicleta,1504,1645.0
7,2015,Pedestre,264,276.0
8,2015,Automóvel,186,227.0
9,2015,Ciclomotor,129,139.0


In [12]:
display(df('''
SELECT l.bairro, count(*) AS acidentes,
       sum(CASE WHEN t.fim_de_semana='Sim' THEN 1 ELSE 0 END) AS no_fim_de_semana
FROM dw.fato_acidente f
JOIN dw.dim_local l USING(sk_local)
JOIN dw.dim_tempo t USING(sk_tempo)
WHERE l.bairro <> 'NÃO INFORMADO'
GROUP BY l.bairro ORDER BY acidentes DESC LIMIT 10'''))
con.close()

,bairro,acidentes,no_fim_de_semana
0,BOA VIAGEM,238,52.0
1,IMBIRIBEIRA,153,36.0
2,SANTO AMARO,135,18.0
3,BOA VISTA,86,13.0
4,CASA AMARELA,85,18.0
5,AFOGADOS,84,26.0
6,MADALENA,84,13.0
7,CORDEIRO,75,15.0
8,CAMPO GRANDE,72,21.0
9,IBURA,71,19.0


## Documentação do dbt
Gera o catálogo/linhagem navegável.
(No Colab, é preciso baixar `target/` ou rodar `dbt docs serve` localmente).

In [13]:
!cd dbt_acidentes && dbt docs generate --profiles-dir . 2>&1 | tail -5
print('Catálogo em dbt_acidentes/target/  (index.html + manifest.json + catalog.json)')

19:28:48  
19:28:48  Concurrency: 4 threads (target='dev')
19:28:48  
19:28:49  Building catalog
19:28:49  Catalog written to /content/dbt_acidentes/target/catalog.json
Catálogo em dbt_acidentes/target/  (index.html + manifest.json + catalog.json)
